## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import shapely

# import kml reading and set supported driver
import fiona
fiona.drvsupport.supported_drivers["KML"] = "rw"

# for parsing HTML inside the Description field
from bs4 import BeautifulSoup

In [ ]:
from gridsample.utils import save_shapefiles

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
PROCESSED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks" / "01 Cleaned Khasras"

In [ ]:
# Function to add suffix to duplicates
def add_suffix_to_duplicates(series):
    counts = {}
    result = []
    for item in series:
        if item in counts:
            counts[item] += 1
            result.append(f"{item}_S{counts[item]}")
        else:
            counts[item] = 0
            result.append(item)
    return result

In [ ]:
# Parse description variables


def description_parser_single_row(html_content):
    # Parse the HTML content
    soup = BeautifulSoup(html_content, "html.parser")

    # Find the inner table containing the attributes
    inner_table = soup.find_all("table")[1]

    # Extract rows from the inner table
    rows = inner_table.find_all("tr")

    # Create a dictionary to store attributes and their values
    data = {}
    for row in rows:
        cols = row.find_all("td")
        if len(cols) == 2:
            key = cols[0].text.strip()
            value = cols[1].text.strip()
            data[key] = value

    return pd.DataFrame([data])


def description_parser(df, description_col_name="Description"):
    # make dataframe of variables
    data = [
        description_parser_single_row(df[description_col_name].values[i])
        for i in range(len(df))
    ]
    df_vars = pd.concat(data)
    df_vars.set_index(df.index, inplace=True)

    return df_vars

## Load raw shapes and process

### Dhar

In [ ]:
# Load dhar khasras
raw_dhar_gdf = gpd.read_file(
    RAW_DATA_DIR / "solar_park_shapefiles" / "Dhar Khasras" / "doc.kml",
    driver="KML"
)

In [ ]:
# remove z-dimension
raw_dhar_gdf.geometry = raw_dhar_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons and only keep the polygon
raw_dhar_gdf = raw_dhar_gdf.explode(index_parts=False)
raw_dhar_gdf = raw_dhar_gdf[raw_dhar_gdf.geometry.type == "Polygon"]

# remove useless Description column
dhar_gdf = raw_dhar_gdf.drop(columns="Description")

In [ ]:
# drop large green shapes (open .KMZ on Google Earth to see)
dhar_gdf = dhar_gdf[dhar_gdf["Name"] != ""]

In [ ]:
# clean up Name so we can separate the villages (string names) from the areas (numbers only)
dhar_gdf["cleaned_name"] = [
    value[0].strip() for value in dhar_gdf["Name"].str.split("/")
]
dhar_gdf["cleaned_name"] = [
    value[0].strip() for value in dhar_gdf["cleaned_name"].str.split(",")
]

In [ ]:
# manual clean
dhar_gdf.loc[dhar_gdf["Name"] == "2829Z1", "cleaned_name"] = "2829"

In [ ]:
# ISOLATE AREA ONLY - select rows that have a number as their Name
dhar_yellow_shapes_gdf = dhar_gdf[dhar_gdf["cleaned_name"].str.isnumeric()]
dhar_yellow_shapes_gdf.plot()

In [ ]:
# ISOLATE VILLAGES ONLY - select rows that have a string as their Name
dhar_village_shapes_gdf = dhar_gdf[~dhar_gdf["cleaned_name"].str.isnumeric()]
dhar_village_shapes_gdf = dhar_village_shapes_gdf.drop(columns="cleaned_name")
dhar_village_shapes_gdf = dhar_village_shapes_gdf.rename(
    columns={"Name": "village_name"}
)
dhar_village_shapes_gdf.plot(column="village_name")

In [ ]:
# add village names to areas
dhar_processed_areas_gdf = dhar_yellow_shapes_gdf.sjoin(
    dhar_village_shapes_gdf, how="left", predicate="intersects"
).drop(columns="index_right")
dhar_processed_areas_gdf.plot(column="village_name")
print("Missing village name:", dhar_processed_areas_gdf["village_name"].isnull().sum())
print("Has village name:", dhar_processed_areas_gdf["village_name"].notnull().sum())

In [ ]:
# Apply the function to the khasra_id column
dhar_processed_areas_gdf["khasra_id"] = dhar_processed_areas_gdf["Name"]
dhar_processed_areas_gdf["khasra_id"] = add_suffix_to_duplicates(dhar_processed_areas_gdf["khasra_id"])

In [ ]:
dhar_processed_areas_gdf.drop(columns="cleaned_name", inplace=True)

In [ ]:
# save to file
save_shapefiles(
    dhar_processed_areas_gdf,
    PROCESSED_DATA_DIR,
    "dhar_cleaned_khasras",
    formats=["kml", "parquet"],
)

### Sagar

In [ ]:
gdfs = []

for filename in ["sagar_khamkuwa", "sagar_mokalpur", "sagar_tekapar"]:
    gdf = gpd.read_file(
        RAW_DATA_DIR / "solar_park_shapefiles" / "Sagar Khasras" / f"{filename}.kml",
        driver="KML",
    )
    gdf["source"] = filename
    gdfs.append(gdf)

raw_sagar_gdf = pd.concat(gdfs, ignore_index=True)

In [ ]:
# remove z-dimension
raw_sagar_gdf.geometry = raw_sagar_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons (each one just contains one polygon anyway)
raw_sagar_gdf = raw_sagar_gdf.explode(column='geometry', ignore_index=True)
raw_sagar_gdf.plot(column="source", legend=True)

In [ ]:
# parse Description variables and merge with shapes dataframe
df_vars = description_parser(raw_sagar_gdf)
raw_sagar_gdf.drop(columns=["Name", "Description"], inplace=True)
sagar_gdf = raw_sagar_gdf.merge(df_vars, left_index=True, right_index=True)

In [ ]:
sagar_gdf.plot(column="PAR_TYPE")

In [ ]:
sagar_gdf["village_name"] = sagar_gdf["source"].str.split("_").str[1]
sagar_gdf["source"] = sagar_gdf["source"].str.split("_").str[0]

In [ ]:
# sagar_gdf["khasra_id"] = "SAGAR_" + sagar_gdf["UNQID"]
sagar_gdf["khasra_id"] = sagar_gdf["village_name"] + "_" + sagar_gdf["KID"]
sagar_gdf["khasra_id"] = add_suffix_to_duplicates(sagar_gdf["khasra_id"])

In [ ]:
save_shapefiles(
    sagar_gdf,
    PROCESSED_DATA_DIR,
    "sagar_cleaned_khasras",
    formats=["kml", "parquet"],
)

### Ashok Nagar

In [ ]:
gdf = gpd.read_file(
    RAW_DATA_DIR / "solar_park_shapefiles" / "Ashok Nagar Khasras" / "Ashok_nagar_khasras.kml",
    driver="KML",
)

In [ ]:
# remove z-dimension
gdf.geometry = gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons (each one just contains one polygon anyway)
# gdf = gdf.explode(column='geometry', ignore_index=True)

In [ ]:
gdf

In [ ]:
gdf.plot(column="Name", legend=True)

In [ ]:
# parse Description variables and merge with shapes dataframe
df_vars = description_parser(gdf)
gdf.drop(columns=["Name", "Description"], inplace=True)
gdf_merged = gdf.merge(df_vars, left_index=True, right_index=True)

In [ ]:
gdf_merged.rename(
    columns={
        "villagee": "village_name",
        "uid": "khasra_id",
        "khasra_no": "Name",
        "Patch_Name": "parcel_id",
    },
    inplace=True,
)

In [ ]:
save_shapefiles(
    gdf_merged,
    PROCESSED_DATA_DIR,
    "ashok_nagar_cleaned_khasras",
    formats=["kml", "parquet"],
)

### Shivpuri

In [ ]:
raw_shivpuri_gdf = gpd.read_file(
    RAW_DATA_DIR / "solar_park_shapefiles" / "Shivpuri Khasras" / "Shivpuri_khasra.kml",
    driver="KML",
)

In [ ]:
# remove z-dimension
raw_shivpuri_gdf.geometry = raw_shivpuri_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons (each one just contains one polygon anyway)
# gdf = gdf.explode(column='geometry', ignore_index=True)

In [ ]:
# parse Description variables and merge with shapes dataframe
df_vars = description_parser(raw_shivpuri_gdf)
raw_shivpuri_gdf.drop(columns=["Name", "Description"], inplace=True)
shivpuri_gdf = raw_shivpuri_gdf.merge(df_vars, left_index=True, right_index=True)

In [ ]:
shivpuri_gdf.plot(column="Patch_Name")

In [ ]:
shivpuri_gdf.plot(column="Cluster", legend=True)

In [ ]:
shivpuri_gdf

In [ ]:
shivpuri_gdf.columns

In [ ]:
shivpuri_gdf.rename(
    columns={
        "villagee": "village_name",
        "uid": "khasra_id",
    },
    inplace=True,
)

In [ ]:
save_shapefiles(
    shivpuri_gdf,
    PROCESSED_DATA_DIR,
    "shivpuri_cleaned_khasras",
    formats=["kml", "parquet"],
)